# crypto_strat_005

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (86).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `crypto` |
| Trading style | `unknown` |
| Timeframe | `unknown` |
| Code cells | 17 |
| Detected functions | calculate_pvs_vms, calculate_vms, calculate_samx, process_ticker, backtest_trades, apply_signals, process_ticker, backtest_trades, calculate_drawdown |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
import pandas as pd
'''
# Skip the first row (the non-header comment line) and load the CSV
data = pd.read_csv('/content/drive/MyDrive/Crypto_Data/Binance_BTCUSDT_merged_minute.csv', skiprows=1)

# Clean up any extra whitespace from column names
data.columns = data.columns.str.strip()

# Rename and select only required columns
data_cleaned = data.rename(columns={
    "Symbol": "Ticker",
    "Date": "Date",
    "Open": "open",
    "High": "high",
    "Low": "low",
    "Close": "close",
    "Volume USDT": "volume"
})[["Ticker", "Date", "open", "high", "low", "close", "volume"]]

# Convert to proper dtypes if needed (optional)
data_cleaned["open"] = data_cleaned["open"].astype(float)
data_cleaned["high"] = data_cleaned["high"].astype(float)
data_cleaned["low"] = data_cleaned["low"].astype(float)
data_cleaned["close"] = data_cleaned["close"].astype(float)
data_cleaned["Volume"] = data_cleaned["volume"].astype(float)


# Display first few rows
data = data_cleaned
data'''

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/Crypto_Data/Binance_BTCUSDT_merged_minute.csv')
data

In [ ]:
data["Ticker"].unique()

In [ ]:
import pandas as pd
'''
# Convert 'Date' column to datetime if it isn't already
#data['Date'] = pd.to_datetime(data['Date'])

# Define the date range
start_date = '2020-01-01'
end_date = '2024-12-31'

# Define the tickers you want to filter
selected_tickers = ['BTCUSDT']  # Add tickers you want

# Apply filtering
data = data[
    (data["Date"] >= start_date) &
    (data["Date"] <= end_date) &
    (data["Ticker"].isin(selected_tickers))
]

# Display filtered results
data'''

In [ ]:
import pandas as pd

# Convert Date to datetime and set as index
data['Date'] = pd.to_datetime(data['Date'])
data.set_index('Date', inplace=True)

# Resample to 5-minute intervals
data = data.resample('5T').agg({
    'Ticker': 'last',                # Last ticker (still BTCUSDT)
    'open': 'first',                 # First open price in the interval
    'high': 'max',                   # Highest price in the interval
    'low': 'min',                    # Lowest price in the interval
    'close': 'last',                 # Last close price in the interval
    'volume': 'sum'                  # Total volume in USDT
}).dropna()  # Drop intervals with incomplete data

# Reset index to get Date back as a column
data.reset_index(inplace=True)

# Save the result
data

In [ ]:
import plotly.express as px
import pandas as pd
'''
# Ensure 'Date' is in Datetime format
data['Date'] = pd.to_datetime(data['Date'])

# Sort by Date
data = data.sort_values('Date')

# Plot Close Prices
fig = px.line(data, x='Date', y='close', title='Close Price Over Time', labels={'close': 'Close Price', 'Date': 'Date'})

# Show the plot
fig.show()'''

In [ ]:
import pandas as pd
import numpy as np

def calculate_pvs_vms(data, atr_period=14, volume_period=14):
    """
    Calculates Price Volatility Strength (PVS) and Volume Momentum Strength (VMS) separately.

    Parameters:
    - data: DataFrame with columns ['Date', 'Ticker', 'close', 'volume']
    - atr_period: Lookback period for ATR calculation (default=14)
    - volume_period: Lookback period for average volume calculation (default=14)

    Returns:
    - UpDated DataFrame with PVS and VMS indicators.
    """

    # Sort data properly
    data = data.sort_values(by=['Ticker', 'Date']).copy()

    # Compute Previous Close
    data['Prev_Close'] = data.groupby('Ticker')['close'].shift(1)

    # Compute True Range (TR) using Close Price Difference (simplified ATR approach)
    data['TR'] = data['TR'] = np.maximum(data['high'] - data['low'],
                            np.maximum(abs(data['high'] - data['Prev_Close']),
                                       abs(data['low'] - data['Prev_Close'])))

    # Compute ATR using rolling mean
    data['ATR'] = data.groupby('Ticker')['TR'].transform(lambda x: x.rolling(atr_period).mean())

    # Compute Average Volume over a rolling window
    data['Avg_Volume'] = data.groupby('Ticker')['volume'].transform(lambda x: x.rolling(volume_period).mean())

    # Compute Price Volatility Strength (PVS)
    data['PVS'] = (data['close'] - data['Prev_Close']) / data['ATR']

    # Compute Volume Momentum Strength (VMS)
    data['VMS'] = (data['volume'] - data['Avg_Volume']) / data['Avg_Volume']

    return data[['Date', 'Ticker', 'open',  'high', 'low', 'close'  ,'volume', 'PVS', 'VMS']]

# Example usage (Assuming you have a DataFrame named 'data' with required columns)
data = calculate_pvs_vms(data)

# Display result
data

In [ ]:
import pandas as pd
'''
def calculate_vms(data, volume_period=14):
    """
    Calculates Volume Momentum Strength (VMS).

    Parameters:
    - data: DataFrame with columns ['date', 'Ticker', 'volume']
    - volume_period: Lookback period for average volume calculation (default=14)

    Returns:
    - Updated DataFrame with VMS indicator.
    """

    # Sort data properly
    data = data.sort_values(by=['Ticker', 'Date']).copy()

    # Compute Average Volume over a rolling window
    data['Avg_Volume'] = data.groupby('Ticker')['volume'].transform(lambda x: x.rolling(volume_period).mean())

    # Compute Volume Momentum Strength (VMS)
    data['VMS'] = (data['volume'] - data['Avg_Volume']) / data['Avg_Volume']

    return data[['Date', 'Ticker', 'open',  'high', 'low', 'volume', 'close', 'PVS', 'VMS']]

# Example usage (Assuming you have a DataFrame named 'data' with required columns)
data = calculate_vms(data)

# Display result
data'''

In [ ]:
import pandas as pd
import numpy as np

def calculate_samx(data, atr_period=14, volume_period=14):
    """
    Calculates the Simplified AMX (SAMX) indicator based on price momentum and volume strength.

    Parameters:
    - data: DataFrame with columns ['Date', 'Ticker', 'open', 'high', 'low', 'close', 'volume']
    - atr_period: Lookback period for ATR calculation (default=14)
    - volume_period: Lookback period for average volume calculation (default=14)

    Returns:
    - Updated DataFrame with SAMX indicator
    """

    # Ensure proper sorting
    data = data.sort_values(by=['Ticker', 'Date']).copy()

    # Compute Previous Close using transform to maintain alignment
    data['prev_close'] = data.groupby('Ticker')['close'].transform(lambda x: x.shift(1))

    # Calculate True Range (TR)
    data['TR'] = np.maximum(data['high'] - data['low'],
                            np.maximum(abs(data['high'] - data['prev_close']),
                                       abs(data['low'] - data['prev_close'])))

    # Compute ATR
    data['ATR'] = data.groupby('Ticker')['TR'].transform(lambda x: x.rolling(atr_period, min_periods=1).mean())

    # Replace zero ATR values with a small constant to prevent division errors
    data['ATR'].replace(0, np.finfo(float).eps, inplace=True)

    # Compute Price Momentum (PMS)
    data['PMS'] = data.groupby('Ticker')['close'].diff() / data['ATR']

    # Compute SAMX
    data['SAMX'] = data['PMS'] * data['VMS']

    return data[['Date', 'Ticker', 'open', 'high', 'low', 'close', 'volume', 'prev_close', 'TR', 'ATR', 'PMS', 'PVS', 'VMS', 'SAMX']]

# Example usage (assuming `data` exists)
data = calculate_samx(data)
data

In [ ]:
# Initialize signal columns
data["signal"] = 0
data["SAMX_signal"] = 0
data["PVS_signal"] = 0
data["VMS_signal"] = 0

# Set SAMX_signal: 1 for bullish (<= 50), 2 for bearish (>= -50)
data.loc[data["SAMX"] <= 50, "SAMX_signal"] = 1
data.loc[data["SAMX"] >= -50, "SAMX_signal"] = 2

# Set PVS_signal: 1 for strong uptrend, 2 for strong downtrend
data.loc[data["PVS"] >= 3, "PVS_signal"] = 1
data.loc[data["PVS"] <= -3, "PVS_signal"] = 2

# Set VMS_signal: 1 for high momentum
data.loc[data["VMS"] >= 1, "VMS_signal"] = 1

# Final Signal (Long)
data.loc[
    (data["SAMX_signal"] == 1) &
    (data["PVS_signal"] == 1) &
    (data["VMS_signal"] == 1),
    "signal"
] = 1

# Final Signal (Short)
data.loc[
    (data["SAMX_signal"] == 2) &
    (data["PVS_signal"] == 2) &
    (data["VMS_signal"] == 1),
    "signal"
] = 2


In [ ]:
data

In [ ]:
data = data.dropna()
data

In [ ]:
#data.to_parquet('/content/drive/MyDrive/5min_data_ADANIENT.parquet')

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from joblib import Parallel, delayed

data["signal"] = 0  # Default value

# Long Entry Condition
data.loc[(data["SAMX"] <= 50) & (data["PVS"] >= 2.25) & (data["VMS"] >= 1.25), "signal"] = 1

# Short Entry Condition (More Strict)
data.loc[(data["SAMX"] >= -50) & (data["PVS"] <= -2.25) & (data["VMS"] >= 1.25), "signal"] = 2

def process_ticker(ticker, ticker_data, initial_capital, brokerage, slippage, tp_pct, sl_pct):
    open_positions = {}
    trades = []
    cumulative_pnl = 0

    for row in ticker_data:
        timestamp, close, open_price, high, low, signal = row["date"], row["close"], row["open"], row["high"], row["low"], row["signal"]

        # Check for open positions and apply exit logic
        if ticker in open_positions:
            position = open_positions[ticker]
            entry_price, direction, qty = position["entry_price"], position["direction"], position["qty"]
            exit_price, exit_reason = None, None

            tp_price = entry_price * (1 + tp_pct) if direction == "long" else entry_price * (1 - tp_pct)
            sl_price = entry_price * (1 - sl_pct) if direction == "long" else entry_price * (1 + sl_pct)

            if (direction == "long" and high >= tp_price) or (direction == "short" and low <= tp_price):
                exit_price, exit_reason = tp_price, "Take Profit"
            elif (direction == "long" and low <= sl_price) or (direction == "short" and high >= sl_price):
                exit_price, exit_reason = sl_price, "Stop Loss"

            if exit_price is not None:
                exit_price *= (1 - slippage) if direction == "long" else (1 + slippage)
                raw_pnl = (exit_price - entry_price) * qty if direction == "long" else (entry_price - exit_price) * qty
                pnl = raw_pnl - (brokerage * initial_capital * 2)
                cumulative_pnl += pnl

                trades.append([ticker, position["entry_time"], timestamp, entry_price, exit_price, pnl, exit_reason, cumulative_pnl, direction])
                del open_positions[ticker]
                continue

        # Open new position (no time restriction)
        if ticker not in open_positions:
            if signal == 1:
                direction = "long"
            elif signal == 2:
                direction = "short"
            else:
                continue

            entry_price = open_price * (1 + slippage) if direction == "long" else open_price * (1 - slippage)
            qty = initial_capital / entry_price
            open_positions[ticker] = {"entry_time": timestamp, "entry_price": entry_price, "direction": direction, "qty": qty}

    return trades

def backtest_trades(data):
    initial_capital = 1000000
    brokerage = 0.001
    slippage = 0.0005
    tp_pct = 0.03
    sl_pct = 0.001

    unique_tickers = data['Ticker'].unique()
    dtype = [
        ("Ticker", "U10"), ("date", "M8[ms]"), ("close", "f8"),
        ("open", "f8"), ("high", "f8"), ("low", "f8"),
        ("PVS", "f8"), ("VMS", "f8"), ("SAMX", "f8"), ("signal", "i4")
    ]

    data_np = np.array(
        list(data[["Ticker", "Date", "close", "open", "high", "low", "PVS", "VMS", "SAMX", "signal"]]
             .itertuples(index=False, name=None)),
        dtype=dtype
    )

    results = Parallel(n_jobs=-1)(delayed(process_ticker)(
        ticker, data_np[data_np['Ticker'] == ticker],
        initial_capital, brokerage, slippage, tp_pct, sl_pct
    ) for ticker in tqdm(unique_tickers))

    trades = [trade for result in results for trade in result]
    return pd.DataFrame(trades, columns=["Ticker", "Entry Time", "Exit Time", "Entry Price", "Exit Price", "PnL", "Exit Reason", "Cumulative PnL", "Direction"])

# Example usage
# data = pd.read_csv("your_data.csv")  # Load dataset
data['Date'] = pd.to_datetime(data['Date'])  # Ensure Date column is in datetime format
tradebook = backtest_trades(data)
tradebook.to_csv("/content/drive/MyDrive/tradebook_BTCUSDT.csv", index=False)
tradebook

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from joblib import Parallel, delayed

# Define parameter grid
param_grid = {
    "samx_thresholds": [70, 90, 100],
    "vms_thresholds": [0.5, 0.75, 1, 1.25, 1.5],
    "pvs_thresholds": [2.1, 2.2, 2.25, 2.3, 2.4, 2.5],
    "sl_pct": [0.001, 0.005, 0.01],  # Stop Loss percentages
    "tp_pct": [0.02, 0.03, 0.04]  # Take Profit percentages
}

# Apply signal logic dynamically
def apply_signals(data, samx_th, vms_th, pvs_th):
    data = data.copy()
    data["signal"] = 0
    data.loc[
        (data["SAMX"] <= samx_th) & (data["PVS"] >= pvs_th) & (data["VMS"] >= vms_th),
        "signal",
    ] = 1
    data.loc[
        (data["SAMX"] >= -samx_th) & (data["PVS"] <= -pvs_th) & (data["VMS"] >= vms_th),
        "signal",
    ] = 2
    return data

def process_ticker(ticker, ticker_data, initial_capital, brokerage, slippage, tp_pct, sl_pct):
    open_positions = {}
    trades = []
    cumulative_pnl = 0

    for row in ticker_data:
        timestamp, close, open_price, high, low, signal = (
            row["date"],
            row["close"],
            row["open"],
            row["high"],
            row["low"],
            row["signal"],
        )

        if ticker in open_positions:
            position = open_positions[ticker]
            entry_price, direction, qty = (
                position["entry_price"],
                position["direction"],
                position["qty"],
            )
            exit_price, exit_reason = None, None

            tp_price = entry_price * (1 + tp_pct) if direction == "long" else entry_price * (1 - tp_pct)
            sl_price = entry_price * (1 - sl_pct) if direction == "long" else entry_price * (1 + sl_pct)

            if (direction == "long" and high >= tp_price) or (direction == "short" and low <= tp_price):
                exit_price, exit_reason = tp_price, "Take Profit"
            elif (direction == "long" and low <= sl_price) or (direction == "short" and high >= sl_price):
                exit_price, exit_reason = sl_price, "Stop Loss"

            if exit_price is not None:
                exit_price *= (1 - slippage) if direction == "long" else (1 + slippage)
                raw_pnl = (exit_price - entry_price) * qty if direction == "long" else (entry_price - exit_price) * qty
                pnl = raw_pnl - (brokerage * initial_capital * 2)
                cumulative_pnl += pnl
                trades.append([ticker, position["entry_time"], timestamp, entry_price, exit_price, pnl, exit_reason, cumulative_pnl, direction])
                del open_positions[ticker]
                continue

        # Entry logic (no stop time constraint)
        if ticker not in open_positions:
            if signal == 1:
                direction = "long"
            elif signal == 2:
                direction = "short"
            else:
                continue

            entry_price = open_price * (1 + slippage) if direction == "long" else open_price * (1 - slippage)
            qty = initial_capital / entry_price
            open_positions[ticker] = {"entry_time": timestamp, "entry_price": entry_price, "direction": direction, "qty": qty}

    return trades

def backtest_trades(data, tp_pct, sl_pct):
    initial_capital = 1000000  # ₹1 lakh
    brokerage = 0.001
    slippage = 0.0005
    unique_tickers = data["Ticker"].unique()

    dtype = [
        ("Ticker", "U10"),
        ("date", "M8[ms]"),
        ("close", "f8"),
        ("open", "f8"),
        ("high", "f8"),
        ("low", "f8"),
        ("PVS", "f8"),
        ("VMS", "f8"),
        ("SAMX", "f8"),
        ("signal", "i4"),
    ]

    data_np = np.array(
        list(
            data[
                ["Ticker", "Date", "close", "open", "high", "low", "PVS", "VMS", "SAMX", "signal"]
            ].itertuples(index=False, name=None)
        ),
        dtype=dtype,
    )

    results = Parallel(n_jobs=-1)(
        delayed(process_ticker)(
            ticker,
            data_np[data_np["Ticker"] == ticker],
            initial_capital,
            brokerage,
            slippage,
            tp_pct,
            sl_pct,
        )
        for ticker in tqdm(unique_tickers, desc="Backtesting tickers")
    )

    trades = [trade for result in results for trade in result]
    return pd.DataFrame(
        trades,
        columns=[
            "Ticker",
            "Entry Time",
            "Exit Time",
            "Entry Price",
            "Exit Price",
            "PnL",
            "Exit Reason",
            "Cumulative PnL",
            "Direction",
        ],
    )


# Drawdown calculator
def calculate_drawdown(pnl_series):
    cumulative = pnl_series.cumsum()
    high_water_mark = cumulative.cummax()
    drawdown = cumulative - high_water_mark
    max_drawdown = drawdown.min()
    return abs(max_drawdown)


# Ensure datetime format for date column
data["Date"] = pd.to_datetime(data["Date"])

# Total iterations count for tqdm progress bar
total_iters = (
    len(param_grid["samx_thresholds"])
    * len(param_grid["vms_thresholds"])
    * len(param_grid["pvs_thresholds"])
    * len(param_grid["sl_pct"])
    * len(param_grid["tp_pct"])
)

# Grid Search Loop with tqdm
with tqdm(total=total_iters, desc="Grid Search") as pbar:
    for samx_th in param_grid["samx_thresholds"]:
        for vms_th in param_grid["vms_thresholds"]:
            for pvs_th in param_grid["pvs_thresholds"]:
                for sl in param_grid["sl_pct"]:
                    for tp in param_grid["tp_pct"]:
                        # Apply entry signal rules
                        data_signaled = apply_signals(data, samx_th, vms_th, pvs_th)

                        # Run backtest
                        trades = backtest_trades(data_signaled, tp_pct=tp, sl_pct=sl)

                        if len(trades) == 0:
                            pbar.update(1)
                            continue

                        total_profit = trades["PnL"].sum()
                        max_drawdown = calculate_drawdown(trades["PnL"])
                        pdr = total_profit / max_drawdown if max_drawdown > 0 else np.nan

                        print(
                            f"Params -> SAMX:{samx_th}, VMS:{vms_th}, PVS:{pvs_th}, SL:{sl*100:.2f}%, TP:{tp*100:.2f}% | "
                            f"Profit: ₹{total_profit:,.2f}, PDR: {pdr:.2f}"
                        )
                        pbar.update(1)
